# pAIr 추론 서버 - RunPod 에서 실제 모델로 띄워 보기

**이 노트북 하나로** 코드 받기 → 라이브러리 설치 → 모델 5개 내려받기·검증 → 서버 실행 → GPU 확인 → 재생 시험까지 합니다.
위에서부터 한 칸씩 `Shift + Enter` 로 실행하세요. (RunPod 의 PyTorch 템플릿 + GPU 1장 Pod 기준, JupyterLab 에서 열기)

- 사진·영상은 **포함되어 있지 않습니다**(응시자·개인 사진은 올리지 않는다). 사진이 없으면 합성 이미지로 속도만 잽니다.
- 끝나면 **Pod 를 Stop/Terminate** 하세요. 켜 두면 과금됩니다.
- 이 노트북은 GPU 에서 **아직 한 번도 돌려 보지 못했습니다.** 막히는 칸이 있으면 그 칸의 출력과 함께 알려 주세요.

## 1. 설정 · GPU 확인

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# 코드 저장소. 비공개라면 아래 REPO_TOKEN 칸에 GitHub 토큰(읽기 전용)을 입력한다. 토큰은 화면·파일에 남기지 않는다.
REPO_URL = "https://github.com/KosAmigo/pAIr-vision.git"
REPO_BRANCH = "main"

WORK = Path("/workspace") if Path("/workspace").exists() else Path.home()
ROOT = WORK / "pAIr-vision"
MODELS = ROOT / "models"
print("작업 폴더:", ROOT)

import shutil

if shutil.which("nvidia-smi"):
    print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                 capture_output=True, text=True).stdout.strip())
else:
    print("nvidia-smi 없음 -> GPU 가 안 잡힌 Pod 일 수 있음(Pod 설정에서 GPU 확인)")
print("python:", sys.version.split()[0])

## 2. 코드 받기
공개 저장소면 그대로 실행됩니다. 비공개면 토큰을 물어봅니다(입력은 화면에 보이지 않음).

In [ ]:
import getpass

if (ROOT / ".git").exists():
    print("이미 있음 -> 최신으로 갱신")
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=False)
else:
    url = REPO_URL
    probe = subprocess.run(["git", "ls-remote", url, REPO_BRANCH], capture_output=True, text=True,
                           env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if probe.returncode != 0:
        token = getpass.getpass("저장소가 비공개입니다. GitHub 토큰(읽기 전용)을 붙여넣으세요: ")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", url, str(ROOT)],
                            capture_output=True, text=True)
    # 토큰이 출력·.git/config 에 남지 않게 정리
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", REPO_URL], check=False)
    print(result.stdout[-300:], result.stderr[-300:].replace(url, "<url>"))
    assert result.returncode == 0, "git clone 실패"
os.chdir(ROOT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. 환경 준비 (라이브러리 · 모델) - 매번 실행해도 안전합니다
`tools/runpod/bootstrap.sh` 가 아래를 확인하고 **없는 것만** 준비합니다. 영구 볼륨(`/workspace`)에 저장하므로 **다음에 Pod 를 켤 때는 대부분 건너뜁니다**(처음 한 번만 수 분~십수 분).
- 시스템 라이브러리(apt: libEGL·libxcb 등) - Pod 를 켤 때마다 사라지므로 빠진 것만 다시 설치(수 초)
- 파이썬 가상환경 `/workspace/venv-vision` (서버·transformers·mediapipe 등. Pod 의 torch 는 그대로 사용)
- 모델 5개(약 700MB): D-FINE-s·OWLv2 는 Hugging Face, MediaPipe 3개는 Google 저장소에서 받고 `models.lock` sha256 으로 검증(다르면 멈춤)

**이 칸에서 오류가 나면 출력 전체를 알려 주세요.** 영구 볼륨이 없는 Pod 면 매번 처음부터 다시 받습니다(Pod 만들 때 Volume 을 붙이세요).

In [ ]:
VENV = WORK / "venv-vision"
VENV_PY = str(VENV / "bin" / "python")
BOOT_ENV = {**os.environ, "WORK": str(WORK)}
boot = subprocess.Popen(["bash", "tools/runpod/bootstrap.sh"], cwd=ROOT, env=BOOT_ENV,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in boot.stdout:
    print(line, end="")
assert boot.wait() == 0, "bootstrap 실패 - 위 출력의 마지막 오류를 확인하세요"

## 4. 서버 실행
API 키·콜백 비밀은 물어봅니다(화면에 보이지 않음). 콜백 주소를 비워 두면 이벤트를 보내지 않고 건수만 셉니다(시험용).

In [ ]:
import getpass
import time
import urllib.request as ur

import secrets

# 비워 두면 무작위로 만든다. 만든 값은 아래 "연결 정보" 칸에서 한 번 보여 준다(백엔드 쪽에 같은 값을 넣어야 한다).
API_KEY = getpass.getpass("서버 API 키(비우면 자동 생성): ") or secrets.token_urlsafe(24)
CALLBACK_URL = input("백엔드 콜백 주소 https://<백엔드>/internal/vision/events (시험만 하면 Enter): ").strip()
CALLBACK_SECRET = ""
env = {**os.environ, "VISION_API_KEY": API_KEY, "VISION_DETECTOR": "real", "VISION_MODELS_DIR": str(MODELS),
       "PYTHONPATH": str(ROOT / "src")}
if CALLBACK_URL:
    env["VISION_CALLBACK_URL"] = CALLBACK_URL
    CALLBACK_SECRET = getpass.getpass("콜백 서명 비밀(백엔드 PAIR_VISION_EVENT_HMAC_SECRET 과 같은 값, 비우면 자동 생성): ") or secrets.token_urlsafe(24)
    env["VISION_CALLBACK_SECRET"] = CALLBACK_SECRET

log = open(WORK / "vision-server.log", "w")
server = subprocess.Popen([VENV_PY, "-m", "pair_vision"], cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT,
                          start_new_session=True)  # 노트북 커널을 다시 시작해도 서버는 계속 돈다

import json
ready = None
for _ in range(60):  # 모델 적재(sha256 검증 포함)에 수십 초
    time.sleep(3)
    if server.poll() is not None:
        break
    try:
        req = ur.Request("http://localhost:8000/readyz", headers={"Authorization": f"Bearer {API_KEY}"})
        ready = json.load(ur.urlopen(req, timeout=3))
        break
    except Exception:
        continue
print(ready or "서버가 안 떴습니다. 로그 마지막 줄:\n" + "".join(open(WORK / "vision-server.log").readlines()[-15:]))
if ready:
    print("\n>>> gpu =", ready["gpu"], "(False 면 GPU 를 못 쓰는 중. 3번 항목 출력의 torch 줄과 Pod 의 GPU 설정을 확인)")

## 5. 재생 시험
`/workspace/photos` 폴더에 시험용 사진(png/jpg)을 올려 두면 그 사진으로, 없으면 합성 이미지로 돌립니다(합성이면 이벤트는 나지 않고 **속도만** 의미 있음).
사진은 휴대폰 시점 책상 사진이어야 손·물체가 잡힙니다. 응시자 실사진은 올리지 마세요.

In [ ]:
photos = WORK / "photos"
if not (photos.is_dir() and any(photos.iterdir())):
    photos = WORK / "synthetic"
    photos.mkdir(exist_ok=True)
    subprocess.run([VENV_PY, "-c", (
        "from PIL import Image, ImageDraw; img = Image.new('RGB', (512, 384), (200, 200, 200)); "
        "ImageDraw.Draw(img).rectangle((120, 200, 380, 330), fill=(60, 60, 60)); img.save(r'%s')" % (photos / "synthetic.jpg"))],
        check=True)
    print("사진이 없어 합성 이미지 사용:", photos)

r = subprocess.run(
    [VENV_PY, "tools/replay/replay.py", "--api-key", API_KEY, "--photos", str(photos),
     "--hold", "30", "--duration", "30", "--settle", "6", "--max-edge", "512"],
    cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-800:])

In [ ]:
# 서버 지표: 분석 지연 분포, 처리/폐기 프레임, 이벤트 수, 모델 오류
req = ur.Request("http://localhost:8000/metrics", headers={"Authorization": f"Bearer {API_KEY}"})
text = ur.urlopen(req, timeout=5).read().decode()
for line in text.splitlines():
    if line.startswith(("pair_vision_frame_latency_seconds_bucket", "pair_vision_frames_", "pair_vision_events_total",
                        "pair_vision_detector_errors_total")) and not line.startswith("#"):
        print(line)
print("\n서버 로그에 '프레임 분석 실패' 가 있으면 모델 오류:",
      sum("프레임 분석 실패" in l for l in open(WORK / "vision-server.log")), "건")

## 6. 외부에서 닿는지 확인 · 백엔드에 알려 줄 연결 정보
RunPod 프록시 주소로 `/healthz` 와 **WebSocket(`/v1/stream`)** 이 통과하는지 확인합니다(프록시가 WebSocket 을 막으면 백엔드가 못 붙습니다).
Pod 에서 HTTP 포트 **8000** 을 공개(Expose HTTP Ports)해 둬야 합니다.

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID", "")
BASE_URL = f"https://{pod_id}-8000.proxy.runpod.net" if pod_id else ""
print("서버 주소(추정):", BASE_URL or "RUNPOD_POD_ID 환경 변수가 없습니다 - Pod 화면의 Connect > HTTP Service 주소를 쓰세요")

if BASE_URL:
    r = subprocess.run([VENV_PY, "tools/runpod/check_proxy.py", BASE_URL, API_KEY], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr[-500:])
    if r.returncode != 0:
        print("-> WebSocket 이 안 되면 프록시가 막는 것일 수 있음(TCP 포트 공개 방식 검토)")

### 백엔드 쪽에 넣을 값 (백엔드 `application.yml` 의 환경 변수 이름)
아래 칸을 실행하면 값이 **한 번 출력**됩니다. 비밀 관리자나 서버 환경 변수로 옮기고, 끝나면 노트북 출력을 지우세요(Edit > Clear Outputs).

| 백엔드 환경 변수 | 값 |
|---|---|
| `PAIR_VISION_INFERENCE_URL` | 위 서버 주소 |
| `PAIR_VISION_INFERENCE_KEY` | 서버 API 키 |
| `PAIR_VISION_EVENT_HMAC_SECRET` | 콜백 서명 비밀 (서버 키 번호 `k1`) |
| `PAIR_SERVER_VISION_ENABLED` | **`false` 로 둔 채** 시작(테스트 회차에서만 켠다) |

**주의: 백엔드의 실제 추론 서버 클라이언트(`/v1/stream`·REST 호출)는 아직 구현되어 있지 않고 가짜(`StubInferenceClient`)만 있습니다.** 그 구현이 들어가기 전까지 백엔드와의 종단 연결은 되지 않습니다. 이 노트북은 서버 단독(5번 항목의 재생 시험)까지 확인합니다.

In [ ]:
print("PAIR_VISION_INFERENCE_URL     =", BASE_URL or "<Pod 화면의 8000 포트 주소>")
print("PAIR_VISION_INFERENCE_KEY     =", API_KEY)
print("PAIR_VISION_EVENT_HMAC_SECRET =", CALLBACK_SECRET or "(콜백 주소를 안 넣어서 없음)")
print("서버 콜백 키 번호: k1")

## 7. 끝내기
```python
server.terminate()
```
**서버를 계속 켜 두고 백엔드와 시험할 때는 아래 칸을 실행하지 마세요.** 시험이 끝나면 아래 칸으로 서버를 끄고, **RunPod 화면에서 Pod 를 Stop/Terminate** 하세요.

결과로 알려 줄 것: 4번 항목의 `gpu` 값, 5번 항목 출력, 지표의 분석 지연(`frame_latency`) 분포.

In [ ]:
server.terminate()
print("서버 종료")